# Ensure ZeroBus & iOS Bootstrap Service Principals

Creates (or finds) two least-privilege service principals for the ZeroBus ingestion pipeline:

1. **ZeroBus SPN** (`dbxw-zerobus-{schema}`) — dedicated to streaming data from the AppKit server to the bronze table via the ZeroBus SDK. Scoped to a single schema and secret scope.
2. **iOS Bootstrap SPN** (`dbxw-ios-bootstrap-{schema}`) — dedicated to authenticating the iOS app to the AppKit auth endpoints before the user signs in with Apple. Granted only `CAN_USE` on the app resource (minimal privilege).

This notebook runs as the **first task** in the `wearables_uc_setup` job. It outputs the ZeroBus SPN's `application_id` as a **task value** so the downstream DDL task can grant it table-level permissions.

**Secret scope keys auto-provisioned by this notebook:**

| Key | Name Variable | Source | Purpose |
| --- | --- | --- | --- |
| Client ID | `client_id_dbs_key` job param | ZeroBus SPN `application_id` | OAuth M2M client identifier |
| Workspace URL | `workspace_url` (fixed) | Derived from config | Databricks workspace URL for SDK/API calls |
| ZeroBus endpoint | `zerobus_endpoint` (fixed) | Derived from workspace ID + region | ZeroBus Ingest server endpoint |
| Target table name | `target_table_name` (fixed) | From job params | Fully qualified bronze table name |
| Stream pool size | `zerobus_stream_pool_size` (fixed) | From job params | SDK gRPC stream pool size |
| JWT signing secret | `jwt_signing_secret_key` job param | Auto-generated (first run only) | HS256 key for app-issued JWTs |
| Apple bundle ID | `apple_bundle_id_key` job param | From job params | iOS app bundle identifier for Apple token validation |
| iOS SPN app ID | `ios_spn_application_id_key` job param | iOS Bootstrap SPN `application_id` | Route guard identity verification |

**Secret scope keys requiring manual admin provisioning:**

| Key | Name Variable | Source | Purpose |
| --- | --- | --- | --- |
| Client secret | `client_secret_dbs_key` job param | Admin-generated | ZeroBus SPN OAuth M2M client secret |

**iOS Bootstrap SPN admin steps (not stored in secret scope):**

| Step | Action | Purpose |
| --- | --- | --- |
| Generate OAuth secret | Workspace UI or CLI | iOS app needs `client_id` + `client_secret` to authenticate via AppKit proxy |
| Grant CAN_USE on app | Apps UI or API | iOS SPN must be able to reach the Databricks App endpoint |
| Embed in iOS binary | Xcode project | The OAuth credentials are compiled into the app |

Key names are **schema-qualified** in dev and hls_fde targets for the ZeroBus SPN (e.g. `client_id_wearables`, `client_secret_wearables`). Auth keys (`jwt_signing_secret`, `apple_bundle_id`, `ios_spn_application_id`) are **not** schema-qualified — they are per-app, not per-schema.

> **Admin steps required after first run:**
> 1. Generate an OAuth secret for the **ZeroBus SPN** and store it under `{client_secret_dbs_key}` in the secret scope
> 2. Generate an OAuth secret for the **iOS Bootstrap SPN** and embed it in the iOS app binary (not stored in scope)
> 3. Grant the **iOS Bootstrap SPN** `CAN_USE` permission on the Databricks App (after app bundle deploy)

**Permissions handled here:**
* Secret scope `READ` ACL for the ZeroBus SPN — so the Databricks App can retrieve all secrets at runtime

**Permissions handled by the DDL task (downstream):**
* `USE CATALOG`, `USE SCHEMA`, `MODIFY`, `SELECT` on the target table


In [0]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

In [0]:
catalog_use = dbutils.widgets.get("catalog_use")
schema_use = dbutils.widgets.get("schema_use")
secret_scope_name = dbutils.widgets.get("secret_scope_name")
client_id_dbs_key = dbutils.widgets.get("client_id_dbs_key")
client_secret_dbs_key = dbutils.widgets.get("client_secret_dbs_key")
zerobus_stream_pool_size = dbutils.widgets.get("zerobus_stream_pool_size")

# JWT Authentication (Phase 1) parameters
jwt_signing_secret_key = dbutils.widgets.get("jwt_signing_secret_key")
apple_bundle_id = dbutils.widgets.get("apple_bundle_id")
apple_bundle_id_key = dbutils.widgets.get("apple_bundle_id_key")
ios_spn_application_id_key = dbutils.widgets.get("ios_spn_application_id_key")

print(f"Catalog:            {catalog_use}")
print(f"Schema:             {schema_use}")
print(f"Secret scope:       {secret_scope_name}")
print(f"Client ID key:      {client_id_dbs_key}")
print(f"Client secret key:  {client_secret_dbs_key}")
print(f"Stream pool size:   {zerobus_stream_pool_size}")
print()
print(f"JWT secret key:     {jwt_signing_secret_key}")
print(f"Apple bundle ID:    {apple_bundle_id}")
print(f"Apple bundle key:   {apple_bundle_id_key}")
print(f"iOS SPN app ID key: {ios_spn_application_id_key}")


In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Naming convention: dbxw-zerobus-{schema} — ties the SPN to this schema
spn_display_name = f"dbxw-zerobus-{schema_use}"
print(f"Target SPN display name: {spn_display_name}")

# Search for an existing SPN with this name
existing_spns = list(
    w.service_principals.list(filter=f'displayName eq "{spn_display_name}"')
)

if existing_spns:
    spn = existing_spns[0]
    is_new_spn = False
    print(f"Found existing SPN: {spn.display_name}")
    print(f"  application_id: {spn.application_id}")
    print(f"  id (workspace): {spn.id}")
    print(f"  active: {spn.active}")
else:
    spn = w.service_principals.create(
        display_name=spn_display_name,
        active=True
    )
    is_new_spn = True
    print(f"Created new SPN: {spn.display_name}")
    print(f"  application_id: {spn.application_id}")
    print(f"  id (workspace): {spn.id}")

spn_application_id = spn.application_id

In [ ]:
# ---------------------------------------------------------------------------
# Find or Create the iOS Bootstrap Service Principal
#
# The iOS bootstrap SPN solves a chicken-and-egg problem: the iOS app
# needs credentials to reach auth endpoints BEFORE the user signs in
# with Apple. By dedicating an SPN with minimal permissions (CAN_USE
# only on the app), the blast radius of compromised iOS-embedded
# credentials is limited to auth endpoints (enforced by the route guard).
#
# Naming convention: dbxw-ios-bootstrap-{schema}
# ---------------------------------------------------------------------------

ios_spn_display_name = f"dbxw-ios-bootstrap-{schema_use}"
print(f"Target iOS SPN display name: {ios_spn_display_name}")

# Search for an existing SPN with this name
existing_ios_spns = list(
    w.service_principals.list(filter=f'displayName eq "{ios_spn_display_name}"')
)

if existing_ios_spns:
    ios_spn = existing_ios_spns[0]
    is_new_ios_spn = False
    print(f"Found existing iOS SPN: {ios_spn.display_name}")
    print(f"  application_id: {ios_spn.application_id}")
    print(f"  id (workspace): {ios_spn.id}")
    print(f"  active: {ios_spn.active}")
else:
    ios_spn = w.service_principals.create(
        display_name=ios_spn_display_name,
        active=True
    )
    is_new_ios_spn = True
    print(f"Created new iOS SPN: {ios_spn.display_name}")
    print(f"  application_id: {ios_spn.application_id}")
    print(f"  id (workspace): {ios_spn.id}")

ios_spn_application_id = ios_spn.application_id


In [0]:
# ---------------------------------------------------------------------------
# Store derived values and SPN identifiers in the secret scope, then
# verify that admin-provisioned secrets are present.
#
# Auto-provisioned (key names from job params — refreshed on every run):
#   {client_id_dbs_key}          — ZeroBus SPN application_id (OAuth M2M client identifier)
#   workspace_url                — Databricks workspace URL
#   zerobus_endpoint             — ZeroBus Ingest server endpoint
#   target_table_name            — Fully qualified bronze table name
#   zerobus_stream_pool_size     — SDK gRPC stream pool size (from job param)
#   {apple_bundle_id_key}        — iOS app bundle identifier (from job param)
#   {ios_spn_application_id_key} — iOS Bootstrap SPN application_id
#
# Auto-provisioned (first run only — preserved on re-runs):
#   {jwt_signing_secret_key}     — HS256 signing key for app-issued JWTs
#                                  NOT regenerated to avoid invalidating existing tokens.
#
# Admin-provisioned:
#   {client_secret_dbs_key}      — ZeroBus SPN OAuth M2M client secret
# ---------------------------------------------------------------------------

# ---- Derive ZeroBus endpoint from workspace metadata ---------------------
# Format: https://<workspace_id>.zerobus.<region>.cloud.databricks.com
# Ref: https://docs.databricks.com/aws/en/ingestion/zerobus-ingest/

workspace_url = w.config.host.rstrip("/")
workspace_id = None

# Try multiple methods to get the workspace ID
try:
    workspace_id = w.get_workspace_id()
except Exception:
    pass

if not workspace_id:
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        workspace_id = ctx.workspaceId().get()
    except Exception:
        pass

if not workspace_id:
    try:
        workspace_id = spark.conf.get("spark.databricks.workspaceUrl").split(".")[0]
    except Exception:
        pass

# Determine the AWS region for the ZeroBus endpoint
try:
    region = spark.conf.get("spark.databricks.clusterUsageTags.region", "")
except Exception:
    region = ""

if not region:
    # Fallback: extract from workspace URL hostname
    # Pattern: <name>.<region>.cloud.databricks.com
    host = workspace_url.replace("https://", "")
    parts = host.split(".")
    if len(parts) == 5 and parts[1] not in ("cloud",):
        region = parts[1]
    else:
        region = "us-east-1"  # AWS default region

zerobus_endpoint = f"https://{workspace_id}.zerobus.{region}.cloud.databricks.com"
target_table = f"{catalog_use}.{schema_use}.wearables_zerobus"

print(f"Workspace URL:     {workspace_url}")
print(f"Workspace ID:      {workspace_id}")
print(f"Region:            {region}")
print(f"ZeroBus endpoint:  {zerobus_endpoint}")
print(f"Target table:      {target_table}")

# ---- Store auto-provisioned values in the secret scope --------------------
try:
    secrets_to_store = {
        client_id_dbs_key:         spn_application_id,
        "workspace_url":           workspace_url,
        "zerobus_endpoint":        zerobus_endpoint,
        "target_table_name":       target_table,
        "zerobus_stream_pool_size": zerobus_stream_pool_size,
        apple_bundle_id_key:       apple_bundle_id,
        ios_spn_application_id_key: ios_spn_application_id,
    }
    for key, value in secrets_to_store.items():
        w.secrets.put_secret(scope=secret_scope_name, key=key, string_value=value)
    print(f"\nStored in scope '{secret_scope_name}':")
    print(f"  {client_id_dbs_key:<30s} = {spn_application_id}")
    print(f"  {'workspace_url':<30s} = {workspace_url}")
    print(f"  {'zerobus_endpoint':<30s} = {zerobus_endpoint}")
    print(f"  {'target_table_name':<30s} = {target_table}")
    print(f"  {'zerobus_stream_pool_size':<30s} = {zerobus_stream_pool_size}")
    print(f"  {apple_bundle_id_key + ':':<30s} = {apple_bundle_id}")
    print(f"  {ios_spn_application_id_key + ':':<30s} = {ios_spn_application_id}")
except Exception as e:
    print(f"\nWarning: could not store secrets (scope may not exist yet): {e}")

# ---- Generate JWT signing secret (first run only) -------------------------
# The JWT signing secret is a cryptographic key — regenerating it on re-runs
# would invalidate all existing access tokens. Only generate if the key
# does not already exist in the scope.
import secrets as secrets_mod
jwt_secret_generated = False
try:
    existing_keys = {s.key for s in w.secrets.list_secrets(scope=secret_scope_name)}
    if jwt_signing_secret_key not in existing_keys:
        jwt_secret = secrets_mod.token_urlsafe(32)
        w.secrets.put_secret(
            scope=secret_scope_name,
            key=jwt_signing_secret_key,
            string_value=jwt_secret,
        )
        jwt_secret_generated = True
        print(f"\n  {jwt_signing_secret_key + ':':<30s} GENERATED (new)")
    else:
        print(f"\n  {jwt_signing_secret_key + ':':<30s} PRESERVED (existing)")
except Exception as e:
    print(f"\nWarning: could not manage JWT signing secret: {e}")

# ---- Check for admin-provisioned client_secret ----------------------------
try:
    existing_keys = {s.key for s in w.secrets.list_secrets(scope=secret_scope_name)}
    credentials_provisioned = client_secret_dbs_key in existing_keys
    print(f"\nSecret scope keys: {sorted(existing_keys)}")
    if credentials_provisioned:
        print(f"ZeroBus OAuth client secret ({client_secret_dbs_key}): PRESENT")
    else:
        print(f"ZeroBus OAuth client secret ({client_secret_dbs_key}): MISSING")
        print()
        print("=" * 65)
        print("ACTION REQUIRED: An admin must provision the client_secret.")
        print("=" * 65)
        print()
        print(f"SPN display name:   {spn_display_name}")
        print(f"SPN application_id: {spn_application_id} (stored as '{client_id_dbs_key}')")
        print(f"SPN workspace ID:   {spn.id}")
        print()
        print("Option 1 \u2014 Workspace UI:")
        print("  Settings > Identity and access > Service principals")
        print(f"  > Select '{spn_display_name}' > Secrets > Generate secret")
        print()
        print("Option 2 \u2014 Databricks CLI (requires account admin):")
        print(f"  databricks account service-principal-secrets create {spn.id}")
        print()
        print("Then store the secret:")
        print(f'  databricks secrets put-secret {secret_scope_name} {client_secret_dbs_key} --string-value "<secret>"')
except Exception:
    credentials_provisioned = False
    print(f"\nCould not check credentials (scope '{secret_scope_name}' may not exist yet).")


In [0]:
from databricks.sdk.service.workspace import AclPermission

# Grant the ZeroBus SPN READ access to the secret scope so the Databricks App
# can retrieve client_id, client_secret, and endpoint credentials at runtime.
# put_acl is idempotent — safe to re-run.
#
# NOTE: The iOS Bootstrap SPN does NOT need scope READ — it authenticates
# via AppKit's proxy (OAuth M2M), not by reading secrets directly. The route
# guard identifies it from proxy-injected headers. Granting scope READ would
# violate least-privilege.
try:
    w.secrets.put_acl(
        scope=secret_scope_name,
        principal=spn_application_id,
        permission=AclPermission.READ
    )
    print(f"Granted READ on scope '{secret_scope_name}' to ZeroBus SPN {spn_application_id}")
except Exception as e:
    # If the scope doesn't exist yet (first deploy before bundle creates it),
    # log the error but don't fail — the scope will be created by the bundle.
    print(f"Warning: Could not set secret scope ACL: {e}")
    print("The secret scope may not exist yet. Re-run after bundle deploy.")


In [0]:
# Pass SPN application_ids to the next task in the job.
# The DDL task reads the ZeroBus SPN via:
#   {{tasks.ensure_service_principal.values.spn_application_id}}
dbutils.jobs.taskValues.set(key="spn_application_id", value=spn_application_id)
dbutils.jobs.taskValues.set(key="ios_spn_application_id", value=ios_spn_application_id)
print(f"Task value set: spn_application_id     = {spn_application_id}")
print(f"Task value set: ios_spn_application_id = {ios_spn_application_id}")


In [0]:
print("=" * 70)
print("Service Principal & Auth Infrastructure Summary")
print("=" * 70)

print()
print("── ZeroBus SPN ─────────────────────────────────────────────────────")
print(f"  Display name:      {spn_display_name}")
print(f"  Application ID:    {spn_application_id}")
print(f"  Newly created:     {is_new_spn}")
print(f"  Scope READ ACL:    granted")

print()
print("── iOS Bootstrap SPN ───────────────────────────────────────────────")
print(f"  Display name:      {ios_spn_display_name}")
print(f"  Application ID:    {ios_spn_application_id}")
print(f"  Newly created:     {is_new_ios_spn}")
print(f"  Scope READ ACL:    NOT granted (least privilege — uses proxy auth)")

print()
print("── Secret Scope ────────────────────────────────────────────────────")
print(f"  Scope name:        {secret_scope_name}")
print(f"    {client_id_dbs_key + ':':<34s} stored (ZeroBus SPN app ID)")
print(f"    {'workspace_url:':<34s} stored")
print(f"    {'zerobus_endpoint:':<34s} stored")
print(f"    {'target_table_name:':<34s} stored")
print(f"    {'zerobus_stream_pool_size:':<34s} stored")
print(f"    {apple_bundle_id_key + ':':<34s} stored ({apple_bundle_id})")
print(f"    {ios_spn_application_id_key + ':':<34s} stored (iOS SPN app ID)")
print(f"    {jwt_signing_secret_key + ':':<34s} {'GENERATED (new)' if jwt_secret_generated else 'PRESERVED (existing)'}")
print(f"    {client_secret_dbs_key + ':':<34s} {'PRESENT' if credentials_provisioned else 'MISSING (admin action required)'}")

print()
print("── Derived Values ──────────────────────────────────────────────────")
print(f"  ZeroBus endpoint:  {zerobus_endpoint}")
print(f"  Target table:      {target_table}")
print(f"  Stream pool size:  {zerobus_stream_pool_size}")
print(f"  Target schema:     {catalog_use}.{schema_use}")

print()
if not credentials_provisioned:
    print(f"*** ADMIN ACTION: {client_secret_dbs_key} must be provisioned    ***")
    print("*** for the ZeroBus SPN before the App can ingest data.       ***")
    print()

if is_new_ios_spn:
    print("*** ADMIN ACTION: iOS Bootstrap SPN is newly created.         ***")
    print("*** Generate an OAuth secret and embed in the iOS app binary. ***")
    print(f"*** Then grant CAN_USE on the Databricks App to:              ***")
    print(f"***   {ios_spn_display_name:<56s} ***")
    print()

print("UC grants (USE CATALOG, USE SCHEMA, MODIFY, SELECT)")
print("will be applied by the downstream DDL task.")
print("=" * 70)
